# Mosaic grids and seam metrics

This notebook demonstrates the **mosaic pipeline**:

1. Building a `MosaicGrid` from a synthetic 3-D volume.
2. Running `fit_mosaic()` to fit one BaSiC model per z-level.
3. Applying the correction with `apply_fit()`.
4. Measuring quality with the **seam-consistency** metrics (`seam_l1`, `seam_pearson`).
5. Using `tune()` for automated Optuna-based hyperparameter search.

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from linum_basic.data import load_sample_image
from linum_basic.fit import apply_fit, fit_mosaic
from linum_basic.metrics import evaluate_correction, seam_l1, seam_pearson
from linum_basic.mosaic import MosaicGrid

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "figure.facecolor": "white",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

## 1. Build a synthetic mosaic volume

We tile the bundled sample image into patches, apply a Gaussian vignette
per z-level (simulating focal-plane drift), and assemble a 3-D mosaic.

In [2]:
TILE = 32  # tile size
GRID = (4, 6)  # 4 rows x 6 cols = 24 tiles
N_Z = 6  # number of z-levels
rng = np.random.default_rng(7)

source = load_sample_image().astype(np.float32) / 255.0

# Build the mosaic array: (Z, H, W)
mosaic_array = np.zeros((N_Z, GRID[0] * TILE, GRID[1] * TILE), dtype=np.float32)

for z in range(N_Z):
    # z-dependent vignette centre shift
    cx = 0.1 * (z - N_Z / 2) / N_Z
    y, x = np.mgrid[-1 : 1 : GRID[0] * TILE * 1j, -1 : 1 : GRID[1] * TILE * 1j]  # type: ignore[misc]
    vignette = (0.5 + 0.5 * np.exp(-1.5 * ((x - cx) ** 2 + y**2))).astype(np.float32)

    # Tile the source; handle boundary by repeating
    reps_y = GRID[0] * TILE // source.shape[0] + 1
    reps_x = GRID[1] * TILE // source.shape[1] + 1
    tiled = np.tile(source, (reps_y, reps_x))
    plane = tiled[: GRID[0] * TILE, : GRID[1] * TILE]

    # Add per-tile brightness variation
    bright = rng.uniform(0.4, 0.8, GRID).repeat(TILE, axis=0).repeat(TILE, axis=1)
    mosaic_array[z] = plane * bright * vignette + 0.03

mosaic = MosaicGrid(mosaic_array, tile_shape=(TILE, TILE), overlap_fraction=0.2)

print(
    f"Mosaic: {mosaic.n_z} z-levels  {mosaic.n_rows}x{mosaic.n_cols} tiles  ({mosaic.n_tiles} total)  tile = {TILE}x{TILE} px"
)
print(f"Seam pairs per z: {len(mosaic.seam_pairs())}")

Mosaic: 6 z-levels  4×6 tiles  (24 total)  tile = 32×32 px


TypeError: object of type 'method' has no len()

In [ ]:
# Visualise the raw mosaic at one z-level
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].imshow(mosaic_array[0], cmap="gray")
axes[0].set_title("Raw mosaic z=0")
axes[1].imshow(mosaic_array[N_Z // 2], cmap="gray")
axes[1].set_title(f"Raw mosaic z={N_Z // 2}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Fit with `fit_mosaic()`

`fit_mosaic()` fits one `BaSiC` model per requested z-level and returns a
`MosaicFit` containing the per-z flat- and dark-fields.

In [ ]:
fit = fit_mosaic(
    mosaic,
    field_mode="per-z",
    basic_kwargs={"working_size": TILE, "estimate_darkfield": True},
    n_workers=1,  # sequential for reproducibility
)

print(f"fit.field_mode  : {fit.field_mode}")
print(f"fit.flatfields  : shape={fit.flatfields.shape}")
print(f"fit.darkfields  : shape={fit.darkfields.shape}")
print(f"fit.z_indices   : {fit.z_indices}")

In [ ]:
# Plot the per-z flatfields
fig, axes = plt.subplots(1, N_Z, figsize=(14, 2.5))
for z, ax in enumerate(axes):
    im = ax.imshow(fit.flatfields[z], cmap="inferno", vmin=0.7, vmax=1.3)
    ax.set_title(f"z={z}", fontsize=8)
    ax.axis("off")
plt.colorbar(im, ax=axes, fraction=0.02, pad=0.04)
plt.suptitle("Estimated flat-fields per z-level", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Apply correction with `apply_fit()`

In [ ]:
corrected = apply_fit(mosaic, fit)

print(f"Corrected volume shape: {corrected.shape}  dtype: {corrected.dtype}")

fig, axes = plt.subplots(2, N_Z, figsize=(14, 5))
for z in range(N_Z):
    axes[0, z].imshow(mosaic_array[z], cmap="gray")
    axes[0, z].set_title(f"Raw z={z}", fontsize=7)
    axes[1, z].imshow(corrected[z].clip(0, 1.5), cmap="gray")
    axes[1, z].set_title(f"Corrected z={z}", fontsize=7)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Seam-consistency metrics

Adjacent tiles physically overlap — after ideal correction the overlapping
pixels should be identical.  `seam_l1` and `seam_pearson` measure
disagreement across those seams.  No ground truth is needed.

In [ ]:
seams = mosaic.seam_pairs()
print(f"Total seam pairs: {len(seams)}")

# Compute metrics on z=0 tiles
raw_tiles = mosaic.iter_tiles(0)
corr_tiles_z0 = corrected[0].reshape(mosaic.n_rows, TILE, mosaic.n_cols, TILE).transpose(0, 2, 1, 3).reshape(-1, TILE, TILE)

print("\n--- z=0 ---")
print(f"Raw    seam_l1    : {seam_l1(raw_tiles, seams):.4f}  (lower is better)")
print(f"Corrected seam_l1  : {seam_l1(corr_tiles_z0, seams):.4f}")
print(f"Raw    seam_pearson: {seam_pearson(raw_tiles, seams):.4f}  (higher is better)")
print(f"Corrected pearson   : {seam_pearson(corr_tiles_z0, seams):.4f}")

# evaluate_correction convenience wrapper
result = evaluate_correction(
    raw_tiles,
    fit.flatfields[0],
    fit.darkfields[0],
    seams,
)
print(f"\nevaluate_correction  : {result}")

## 5. Global vs per-z mode

`field_mode='global'` averages all per-z fields into a single estimate —
more robust on noisy data, but blurs depth-dependent changes.

In [ ]:
fit_global = fit_mosaic(
    mosaic,
    field_mode="global",
    basic_kwargs={"working_size": TILE},
    n_workers=1,
)

print(f"Global flatfield shape: {fit_global.flatfields.shape}")

corrected_global = apply_fit(mosaic, fit_global)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(fit.flatfields.mean(axis=0), cmap="inferno", vmin=0.7, vmax=1.3)
axes[0].set_title("Per-z (mean)")
axes[1].imshow(fit_global.flatfields, cmap="inferno", vmin=0.7, vmax=1.3)
axes[1].set_title("Global")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Automated tuning with `tune()`

`tune()` uses Optuna to minimise the seam-L1 metric over the BaSiC
hyperparameter space (`working_size`, `l_s_divisor`, `l_d_divisor`,
`epsilon`, `estimate_darkfield`).

> **Note:** `optuna` must be installed (`uv sync --extra notebooks`).

In [ ]:
try:
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _has_optuna = True
except ImportError:
    _has_optuna = False
    print("optuna not installed — skipping tuning demo.")
    print("Install with: uv sync --extra notebooks")

In [ ]:
if _has_optuna:
    from linum_basic.tuning import tune

    result = tune(
        mosaic,
        n_trials=10,  # quick demo; use ≥50 for real data
        z_subsample=2,  # evaluate on 2 z-levels per trial
        max_tiles=None,  # use all tiles (small grid)
        n_workers=1,
        seed=0,
    )

    print("Best params:")
    for k, v in result.best_params.items():
        print(f"  {k:25s} = {v}")
    print(f"\nBest seam-L1: {result.best_value:.6f}")

In [ ]:
if _has_optuna and result.trials_df is not None:
    df = result.trials_df.sort_values("value")
    print(
        df[["value", "params_working_size", "params_l_s_divisor", "params_estimate_darkfield"]].head(5).to_string(index=False)
    )

## Next steps

* **Advanced** — GPU backend, parallel z-levels, zarr I/O: see the
  [Advanced](04_advanced.ipynb) notebook.
* **CLI** — use `basic fit` and `basic tune` for batch jobs from the
  command line.